# Probe 023 launcher (phase C)
Colab is a compute worker only. This driver kernel NEVER imports the model stack -- `run.py` runs as a child process, so no kernel restart is ever needed.

**One-time setup:** create a fine-grained GitHub PAT scoped to this single repository, Contents: Read and write, with an expiry. In Colab: key icon (Secrets) -> add `SCOUT_RESULTS_PAT` -> enable notebook access. The PAT never appears in this notebook or its output.

Results branch (contract-bound): `results/probe-023-349af5ad0b3e`

Per session: run all cells top to bottom. After a disconnect, rerun all cells -- run.py resumes from the bundle on Drive, and the transport cell pushes whatever is new.

In [ ]:
PHASE = 'C'
REPO_URL = 'https://github.com/Moroseui/concept-research-scout.git'
PIN_COMMIT = '98a6f68b17a4e963334c6ab32b9e6ed23b488810'
RESULTS_BRANCH = 'results/probe-023-349af5ad0b3e'
OUTPUT_DIR = '/content/drive/MyDrive/concept-research-scout-results/023_C'
PHASE_S_DIR = '/content/drive/MyDrive/concept-research-scout-results/023_v2'

In [ ]:
from google.colab import drive, userdata
import os, shutil, time
MP = '/content/drive'
def _mounted(mp):
    try:
        return any(len(l.split()) > 1 and l.split()[1] == mp
                   for l in open('/proc/mounts'))
    except OSError:
        return False
try:
    if os.path.isdir(MP) and not _mounted(MP) and os.listdir(MP):
        stale = f'/content/drive_stale_{int(time.time())}'
        shutil.move(MP, stale)
        print('stale mountpoint residue moved to', stale,
              '(a crashed FUSE mount left a corpse on a surviving VM)')
except OSError as e:
    print('could not inspect/move mountpoint residue:', e,
          '-- if the mount below fails, Runtime > Disconnect and delete runtime')
drive.mount(MP, force_remount=True)
GH_PAT = userdata.get('SCOUT_RESULTS_PAT')  # never printed
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')  # inherited by the run.py child; never printed

In [ ]:
%cd /content
!rm -rf /content/scout-repo
!git clone {REPO_URL} /content/scout-repo
%cd /content/scout-repo
!git checkout {PIN_COMMIT}

In [ ]:
!pip install -q -r probes/023/requirements.txt

In [ ]:
# --- origin_direct staging: the pinned Zenodo record is the source;
# the Drive mount carries ONLY small inputs/outputs (FUSE is out of
# the 99 GB path entirely -- seven recorded casualties).
import os, json, urllib.request
LOCAL = '/content/work'
os.makedirs(LOCAL, exist_ok=True)
RECORD_JSON = LOCAL + '/zenodo_record.json'
with urllib.request.urlopen('https://zenodo.org/api/records/16813698') as r:
    rec = json.load(r)
assert str(rec['id']) == '16813698', 'server returned a different record than the declared pin'
json.dump(rec, open(RECORD_JSON, 'w'), indent=2)
_a = [f for f in rec['files'] if f['key'].endswith('.7z')]
assert len(_a) == 1, _a
ARCHIVE_LOCAL = LOCAL + '/' + _a[0]['key']
ARCHIVE_URL = _a[0]['links']['self']
print('pinned record', rec['id'], _a[0]['key'], round(_a[0]['size']/1e9, 1), 'GB (origin_direct)')

In [ ]:
import hashlib, subprocess
_name = os.path.basename(ARCHIVE_LOCAL)
_entries = [f for f in rec['files'] if f.get('key') == _name]
assert len(_entries) == 1
_ck = _entries[0].get('checksum', '')
if not _ck.startswith('md5:'):
    raise SystemExit('driver configuration error: pinned record supplies no '
                     'md5 for ' + _name + '; refusing a 99 GB staging pass')
EXPECT_MD5 = _ck.split(':', 1)[1]
EXPECT_SIZE = _entries[0]['size']
def _md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 22), b''):
            h.update(chunk)
    return h.hexdigest()
subprocess.run(['apt-get', '-qq', 'install', '-y', 'aria2'], check=False)
LOCAL_DATA = LOCAL + '/extracted'
_done = False
if os.path.exists(ARCHIVE_LOCAL):
    print('verifying existing local archive md5 (~4 min)...')
    if os.path.getsize(ARCHIVE_LOCAL) == EXPECT_SIZE and _md5(ARCHIVE_LOCAL) == EXPECT_MD5:
        _done = True
    else:
        print('existing local archive fails integrity; removing')
        os.remove(ARCHIVE_LOCAL)
if not _done:
    for _attempt in (1, 2):
        _part = _name + '.part'
        print('downloading from Zenodo (attempt', _attempt, 'of 2; 16-way aria2c;',
              round(EXPECT_SIZE/1e9, 1), 'GB -- ETA prints below)...')
        _rc = subprocess.run(['aria2c', '-x16', '-s16', '-k4M',
                              '--file-allocation=none', '-c',
                              '-d', LOCAL, '-o', _part, ARCHIVE_URL]).returncode
        _pp = LOCAL + '/' + _part
        if (_rc == 0 and os.path.exists(_pp)
                and os.path.getsize(_pp) == EXPECT_SIZE):
            print('verifying downloaded bytes md5 (~4 min)...')
            if _md5(_pp) == EXPECT_MD5:
                os.replace(_pp, ARCHIVE_LOCAL)
                _done = True
                break
        print('download failed integrity/transfer (rc', _rc, '); discarding partial')
        for _f in (_pp, _pp + '.aria2'):
            if os.path.exists(_f):
                os.remove(_f)
if not _done:
    raise SystemExit('ORIGIN_DOWNLOAD_INTEGRITY_FAILURE: two direct downloads '
                     'from the pinned record failed; check Zenodo status and '
                     'network, then rerun')
print('local archive verified: md5', EXPECT_MD5)
print('local archive bytes:', os.path.getsize(ARCHIVE_LOCAL))

In [ ]:
SUFFIXES = ['_space-ncct_cbf.nii.gz', '_space-ncct_cbv.nii.gz', '_space-ncct_mtt.nii.gz', '_space-ncct_tmax.nii.gz', '_lesion-msk.nii.gz', '_ncct.nii.gz']
if not os.path.isdir(LOCAL_DATA):
    !apt-get -qq install -y p7zip-full
    _inc = ' '.join('-ir!*' + x for x in SUFFIXES)
    !7z x "{ARCHIVE_LOCAL}" -o"{LOCAL_DATA}" {_inc} -y
_n = sum(len(f) for _, _, f in os.walk(LOCAL_DATA))
print('extracted files (local):', _n)
assert _n >= 800, 'extraction incomplete -- refusing to reach the census'

In [ ]:
# Console (incl. any crash traceback) persists to Drive; refresh-proof.
!mkdir -p {OUTPUT_DIR}
!python probes/023/run.py --phase {PHASE} --output-dir {OUTPUT_DIR} --data-dir {LOCAL_DATA} --archive-file {ARCHIVE_LOCAL} --record-json {RECORD_JSON} --phase-s-dir {PHASE_S_DIR} 2>&1 | tee -a {OUTPUT_DIR}/driver_console.log

In [ ]:
# E1 transport: mirror the bundle onto the contract-bound results
# branch. ORDER MATTERS: check out the branch FIRST, then overlay the
# bundle (copy-then-checkout fails after session 1: git refuses to
# overwrite untracked files the branch already tracks). The PAT rides
# in a header, never in argv or output.
import shutil, subprocess, pathlib, base64, datetime
repo = pathlib.Path('/content/scout-repo')
dest = repo / 'probes/023/results_v2'
def git(*a, **k):
    r = subprocess.run(['git', *a], cwd=repo, capture_output=True, text=True, **k)
    if r.returncode: raise SystemExit(f'git {a[0]} failed: {r.stderr[-400:]}')
    return r.stdout
git('config', 'user.email', 'colab-runner@scout.local')
git('config', 'user.name', 'scout colab runner')
auth = base64.b64encode(f'x-access-token:{GH_PAT}'.encode()).decode()
hdr = f'http.extraheader=AUTHORIZATION: basic {auth}'
if subprocess.run(['git', '-c', hdr, 'fetch', 'origin', RESULTS_BRANCH], cwd=repo, capture_output=True).returncode == 0:
    git('checkout', '-B', RESULTS_BRANCH, f'origin/{RESULTS_BRANCH}')
else:
    git('checkout', '-B', RESULTS_BRANCH, PIN_COMMIT)
if dest.exists(): shutil.rmtree(dest)
shutil.copytree(OUTPUT_DIR, dest)
git('add', '-f', 'probes/023/results_v2')
stamp = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
subprocess.run(['git', 'commit', '-m', f'session results {stamp}'], cwd=repo, capture_output=True)
git('-c', hdr, 'push', 'origin', RESULTS_BRANCH)
print('pushed', RESULTS_BRANCH)

When `run.py` reports the study complete, the results-validate workflow on the pushed branch verifies the bundle and opens the record-result PR. Merging that PR is the human gate.